# Model 1
```
Model 1: Image-Only Multi-Output Biomass Regression
Input : pasture image
Output: 5 biomass targets
        [Dry_Green_g, GDM_g, Dry_Total_g]
```

In [18]:
import os
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import matplotlib.pyplot as plt

class Config:
    # paths
    CSV_PATH      = "train.csv"          # path to train.csv
    IMAGE_DIR     = ""                   # root dir; image_path in CSV is relative to this
    SAVE_PATH     = "model1_best.pth"    # where to save best weights

    # targets — remove any you don't want to predict
    # TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    TARGETS = ["Dry_Green_g", "GDM_g", "Dry_Total_g"]

    # training
    IMAGE_SIZE  = 224
    BATCH_SIZE  = 16
    NUM_EPOCHS  = 120
    LR          = 1e-4
    WEIGHT_DECAY= 1e-4
    VAL_SPLIT   = 0.2
    SEED        = 42

    # model
    BACKBONE    = 'resnet34' #"efficientnet_b0"   # or "resnet50", "resnet34"
    PRETRAINED  = True
    DROPOUT     = 0.3

    # device
    DEVICE = "cuda" #if torch.cuda.is_available() else "cpu"

In [19]:

def prepare_data(csv_path, targets, val_split=0.2, seed=42):
    """
    Reads train.csv (long format: 5 rows per image) and
    pivots to wide format (1 row per image, 5 target columns).
    Then splits into train/val by IMAGE (not by row).
    """
    df = pd.read_csv(csv_path)

    # pivot: one row per image
    df_wide = df.pivot_table(
        index=["image_path"],
        columns="target_name",
        values="target"
    ).reset_index()
    df_wide.columns.name = None

    # keep only targets we care about
    available = [t for t in targets if t in df_wide.columns]
    missing   = [t for t in targets if t not in df_wide.columns]
    if missing:
        print(f"[WARNING] These targets not found in data: {missing}")

    df_wide = df_wide[["image_path"] + available].dropna()
    print(f"Dataset: {len(df_wide)} images, targets: {available}")

    # train/val split by image
    train_df, val_df = train_test_split(
        df_wide, test_size=val_split, random_state=seed
    )
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")

    return train_df, val_df, available

cfg = Config()
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)
print(f"Device: {cfg.DEVICE}")

train_df, val_df, targets = prepare_data(
    cfg.CSV_PATH, cfg.TARGETS, cfg.VAL_SPLIT, cfg.SEED
)
targets

Device: cuda
Dataset: 357 images, targets: ['Dry_Green_g', 'GDM_g', 'Dry_Total_g']
Train: 285 | Val: 72


['Dry_Green_g', 'GDM_g', 'Dry_Total_g']

In [20]:
class BiomassDataset(Dataset):
    """
    Loads one row per IMAGE (not per target).
    Each image has all 5 target values as a vector.
    """
    def __init__(self, df_wide, image_dir, targets, transform=None):
        """
        df_wide  : DataFrame with one row per image, columns = targets
        image_dir: root directory for image paths
        targets  : list of target column names
        transform: torchvision transforms
        """
        self.df        = df_wide.reset_index(drop=True)
        self.image_dir = image_dir
        self.targets   = targets
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row       = self.df.iloc[idx]
        img_path  = os.path.join(self.image_dir, row["image_path"])
        image     = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[self.targets].values.astype(np.float32),
            dtype=torch.float32
        )
        return image, labels

def get_transforms(image_size, mode="train"):
    """
    train : augmentations to combat overfitting (only 357 images!)
    val   : just resize + normalize
    """
    mean = [0.485, 0.456, 0.406]   # ImageNet stats (backbone was pretrained on it)
    std  = [0.229, 0.224, 0.225]

    if mode == "train":
        return transforms.Compose([
            transforms.Resize((image_size + 32, image_size + 32)),
            transforms.RandomCrop(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                   saturation=0.3, hue=0.1),
            transforms.RandomRotation(30),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        # if val/test, no augmentation, just resize + normalize
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
train_dataset = BiomassDataset(
    train_df, cfg.IMAGE_DIR, targets,
    transform=get_transforms(cfg.IMAGE_SIZE, "train")
)
val_dataset = BiomassDataset(
    val_df, cfg.IMAGE_DIR, targets,
    transform=get_transforms(cfg.IMAGE_SIZE, "val")
)

train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE,
                            shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.BATCH_SIZE,
                            shuffle=False, num_workers=2, pin_memory=True)

In [21]:
class BiomassModel(nn.Module):
    """
    Pretrained CNN backbone with a custom regression head.
    Outputs num_targets values (one per biomass component).
    """
    def __init__(self, backbone_name, num_targets, pretrained=True, dropout=0.3):
        super().__init__()

        # ── backbone ──────────────────────────────────
        if backbone_name == "efficientnet_b0":
            weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
            backbone = models.efficientnet_b0(weights=weights)
            in_features = backbone.classifier[1].in_features
            backbone.classifier = nn.Identity()            # remove original head

        elif backbone_name == "resnet50":
            weights = models.ResNet50_Weights.DEFAULT if pretrained else None
            backbone = models.resnet50(weights=weights)
            in_features = backbone.fc.in_features
            backbone.fc = nn.Identity()

        elif backbone_name == "resnet34":
            weights = models.ResNet34_Weights.DEFAULT if pretrained else None
            backbone = models.resnet34(weights=weights)
            in_features = backbone.fc.in_features
            backbone.fc = nn.Identity()

        else:
            raise ValueError(f"Unknown backbone: {backbone_name}")

        self.backbone = backbone

        # ── regression head ───────────────────────────
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_targets),
            nn.ReLU()          # biomass is always >= 0
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)
    
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        preds = model(images)
        loss  = criterion(preds, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)

@torch.no_grad()
def evaluate(model, loader, criterion, device, targets):
    model.eval()
    total_loss  = 0.0
    all_preds   = []
    all_labels  = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images)
        loss  = criterion(preds, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    # per-target R²
    r2_scores = {}
    for i, t in enumerate(targets):
        ss_res = np.sum((all_labels[:, i] - all_preds[:, i]) ** 2)
        ss_tot = np.sum((all_labels[:, i] - all_labels[:, i].mean()) ** 2)
        r2_scores[t] = 1 - ss_res / (ss_tot + 1e-8)

    avg_loss = total_loss / len(loader.dataset)
    return avg_loss, r2_scores, all_preds, all_labels

# ─────────────────────────────────────────────
# PLOTTING
# ─────────────────────────────────────────────
def plot_training_curves(train_losses, val_losses, save_path="training_curves.png"):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses,   label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.title("Training Curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Saved training curves → {save_path}")


def plot_predictions(all_preds, all_labels, targets, save_path="predictions.png"):
    n = len(targets)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]

    for i, (ax, t) in enumerate(zip(axes, targets)):
        ax.scatter(all_labels[:, i], all_preds[:, i], alpha=0.5, s=20)
        lims = [
            min(all_labels[:, i].min(), all_preds[:, i].min()),
            max(all_labels[:, i].max(), all_preds[:, i].max()),
        ]
        ax.plot(lims, lims, "r--", linewidth=1)
        ax.set_xlabel("Ground Truth (g)")
        ax.set_ylabel("Predicted (g)")
        ax.set_title(t)

    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()
    print(f"Saved prediction plots → {save_path}")

In [22]:
model = BiomassModel(
    backbone_name=cfg.BACKBONE,
    num_targets=len(targets),
    pretrained=cfg.PRETRAINED,
    dropout=cfg.DROPOUT
).to(cfg.DEVICE)

print(f"\nModel: {cfg.BACKBONE} | Targets: {len(targets)}")

# ── 3. Training setup ─────────────────────
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.NUM_EPOCHS, eta_min=1e-6
)

# ── 4. Training loop ──────────────────────
train_losses, val_losses = [], []
best_val_loss = float("inf")

# ── 4. Training loop ──────────────────────
train_losses, val_losses = [], []
best_val_loss = float("inf")

for epoch in range(1, cfg.NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, cfg.DEVICE)
    val_loss, r2_scores, val_preds, val_labels = evaluate(
        model, val_loader, criterion, cfg.DEVICE, targets
    )
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    # print every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        r2_str = " | ".join([f"{t}: {v:.3f}" for t, v in r2_scores.items()])
        print(f"Epoch {epoch:3d}/{cfg.NUM_EPOCHS} | "
                f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
        print(f"  R² → {r2_str}")

    # save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "optimizer":   optimizer.state_dict(),
            "targets":     targets,
            "config":      {
                "backbone":   cfg.BACKBONE,
                "image_size": cfg.IMAGE_SIZE,
                "dropout":    cfg.DROPOUT,
            }
        }, cfg.SAVE_PATH)

print(f"\nBest val loss: {best_val_loss:.4f} → saved to {cfg.SAVE_PATH}")

# ── 5. Final evaluation on best model ─────
checkpoint = torch.load(cfg.SAVE_PATH, map_location=cfg.DEVICE)
model.load_state_dict(checkpoint["model_state"])
_, r2_scores, val_preds, val_labels = evaluate(
    model, val_loader, criterion, cfg.DEVICE, targets
)

print("\n── Final Val R² Scores ──")
for t, r2 in r2_scores.items():
    print(f"  {t:<20s}: {r2:.4f}")
    
plot_training_curves(train_losses, val_losses)
plot_predictions(val_preds, val_labels, targets)

5.7%

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /home/ali/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100.0%



Model: resnet34 | Targets: 3
Epoch   1/120 | Train Loss: 2013.7979 | Val Loss: 1564.3492
  R² → Dry_Green_g: -1.144 | GDM_g: -1.862 | Dry_Total_g: -2.964
Epoch   5/120 | Train Loss: 794.7584 | Val Loss: 568.7142
  R² → Dry_Green_g: 0.056 | GDM_g: -0.106 | Dry_Total_g: -0.254
Epoch  10/120 | Train Loss: 333.5464 | Val Loss: 293.9997
  R² → Dry_Green_g: 0.459 | GDM_g: 0.390 | Dry_Total_g: 0.426
Epoch  15/120 | Train Loss: 275.7707 | Val Loss: 271.4845
  R² → Dry_Green_g: 0.567 | GDM_g: 0.475 | Dry_Total_g: 0.385
Epoch  20/120 | Train Loss: 319.8836 | Val Loss: 175.5303
  R² → Dry_Green_g: 0.704 | GDM_g: 0.697 | Dry_Total_g: 0.585
Epoch  25/120 | Train Loss: 294.9155 | Val Loss: 235.2535
  R² → Dry_Green_g: 0.580 | GDM_g: 0.534 | Dry_Total_g: 0.512
Epoch  30/120 | Train Loss: 280.1328 | Val Loss: 250.4961
  R² → Dry_Green_g: 0.575 | GDM_g: 0.477 | Dry_Total_g: 0.485
Epoch  35/120 | Train Loss: 221.1761 | Val Loss: 186.5506
  R² → Dry_Green_g: 0.631 | GDM_g: 0.660 | Dry_Total_g: 0.618
Epo